# Train your Aimodel writer (a tiny transformer)

This notebook trains the transformer that words your Aimodel's answers. Your data
stays in your Kaggle account; the trained writer runs on your phone with numpy only.

**Before you start**
1. In Aimodel, type `/export`. It creates a `writer_data` folder (3 files).
2. On Kaggle: **Create → New Dataset**, upload those 3 files, and keep it **Private**.
3. In this notebook: **Add Input** → your dataset. In **Settings**: Accelerator **GPU T4 x2**
   (or P100), and **Internet on** (only needed to download the Aimodel code).
4. **Run all**. Training takes about 10–20 minutes on a GPU.

In [ ]:
!git clone -q -b main https://github.com/iamsabroy-gif/Aimodel.git
%cd Aimodel

In [ ]:
import glob, os
found = glob.glob("/kaggle/input/**/writer_data.jsonl", recursive=True)
DATA = os.path.dirname(found[0]) if found else "/kaggle/working/writer_data"
print("Training data:", DATA if found else "not found - add your dataset as an input (step 3)")

## Train

`--size base` is about 3 million parameters. To **keep training a writer you already have**
(the gradual way: export again after teaching your model more), add it as an input too and
add `--resume /kaggle/input/<its dataset>/writer.npz`.

In [ ]:
!python -m aimodel.train_writer --data "$DATA" --out /kaggle/working/writer.npz --size base --steps 6000

## Try it here

In [ ]:
from aimodel.transformer import TinyTransformer
from aimodel.writer import grounded
writer = TinyTransformer.load("/kaggle/working/writer.npz")
evidence = ["Cats are mammals.", "Mammals are warm-blooded animals."]
draft = writer.write("Is a cat an animal?", evidence)
print(draft)
print("passes the evidence check:", grounded(draft, evidence + ["Is a cat an animal?"])[0])

## Take it home

Download `writer.npz` from the **Output** panel (`/kaggle/working`). Put it next to your
brain on your phone and type `/writer load writer.npz`. Every answer it drafts is checked
against the evidence; drafts that aren't fully supported fall back to the template answer.